<a href="https://www.kaggle.com/code/anamkhan001/khan-airline-delay?scriptVersionId=245934920" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Airline Delay ML Project: Objectives

The purpose of this project is get a solid understanding of the airline delay data set in order to find out:

* Which airport has perfromed the worst?
* Which airline has performed the worst?
* How many flights will have delays 15 mins or more during a specific month/year, based on the airline and airport

Key metrics I'm looking at per airport and airline:
* Arrival delay rate
* cancellation and diversion rate
* Average delay time
  

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import pdb
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import mean_absolute_error, accuracy_score, mean_squared_error, mean_absolute_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor



# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Load input data
df = pd.read_csv("/kaggle/input/airline-delay/Airline_Delay_Cause.csv")
print(df.head())
print(df.shape)


At this point we want to take a closer look at our data and understand what each row is trying to describe. 

Each row is characterizing the amount of delays and their causes as well as the length of the delay and causes. It's describig this not per flight, but for a specific airline and airport combination for a specific month at a time. All the *_ct variables sum up and are equal to the arr_dl15 value. The same is for the _delay columns to arr_delay. Cancellations and diversions are treated seperately from delays. Let's take a look at one of the rows of data to get a better understanding. 

Another important thing to note is that the dates are not stored in the datetime format, something to cleanup.

In [ ]:
df.iloc[0]

# Data Cleaning
* Missing cells
* Incorrect data types
* Duplicate rows
* etc.

In [ ]:
# Figure out the percentage of missing cells to total cells
missing = df[df.isnull().any(axis=1)]
missing_sum = df.isnull().sum()
total_missing = missing_sum.sum()
total_cells = np.prod(df.shape)
percent_cells_missing = (total_missing/total_cells) * 100
print("Percent Missing: ", percent_cells_missing)

# Figure out how many % of rows are removed
df_dropped = df.dropna()
percent_rows_dropped = ( df_dropped.shape[0] / df.shape[0] ) * 100
print("Percent of original df after rows dropped: ", percent_rows_dropped)

# Does the dropped data effect specific carriers more than others?
old_carriers = df.carrier_name.value_counts()
new_carriers = df_dropped.carrier_name.value_counts()
percent_change_carriers =  ( (old_carriers - new_carriers) / old_carriers ) * 100

# Does the dropped data effect specific airports more than others?
old_airports = df.airport_name.value_counts()
new_airports = df_dropped.airport_name.value_counts()
percent_change_airports =  ( (old_airports - new_airports) / old_airports ) * 100
percent_change_airports = percent_change_airports[percent_change_airports > 1]
print("Airport with most change: ", percent_change_airports.idxmax())
print("Airport change percentage: ", percent_change_airports.max())


In [ ]:
# Worst is for Mobile (AL), a 8.5% drop, I think it's ok
df = df_dropped

# More Data Cleaning - Duplicate rows?
dupes = df.duplicated().sum()
print("Duplicated rows: ", dupes) # None, so good to move on

# More Data Cleaning - set up dates in correct time format
df['combined_date'] = pd.to_datetime(df['month'].astype(str) + '-' + df['year'].astype(str), format='%m-%Y') 
df = df.drop(['month', 'year'], axis=1)
df = df[['combined_date'] + [col for col in df.columns if col != 'combined_date']]
df['Year'] = df['combined_date'].dt.year
df['Month'] = df['combined_date'].dt.month

# Data Visualizations

In [ ]:
# Heatmap
pivot = df.pivot_table(index='carrier_name', columns='Year', values='arr_del15', aggfunc='mean')
plt.figure(figsize=(12,8))
plt.title('Average Number of Delayed Flights')
sns.heatmap(data=pivot, annot=True, cmap='Reds')

# Line Plot
plt.figure(figsize=(12,8))
sns.lineplot(data=df, x='Year', y='arr_del15', hue='carrier')  # Multiple groups
plt.tight_layout()

# Plot stacked bars of scaled components
delay_components = df.groupby('carrier_name')[['carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct']].sum()
delay_components.plot(
    kind='barh',
    stacked=True,
    figsize=(10, 6),
)
plt.title('Breakdown of Delay causes')
plt.xlabel('Delay Contribution')
plt.ylabel('Airline')
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()


#### Key takeaways:
* Obvious that Southwest Airlines has the most delay issues
* During 2020 everyones number of delays went down - COVID related?
* late aircraft seems to effect delay times the most, along with carrier issues

Is ExpressJet Airlines duplicated? One entry is as a LLC and the other as Inc?

A quick google search shows that the ExpressJet did indeed switch ownership, so probably useful to show performace for both companies. 

In [ ]:
# Scale attributes and generate composite score

def group_delay_scaled(feature, df):
    group = df.groupby(feature).agg(
        TOTAL_FLIGHTS_count = ('arr_flights', 'sum'),
        TOTAL_ARR_DELAY_count = ('arr_del15', 'sum'),
        AVG_ARR_DELAY_mins = ('arr_delay', 'mean'),
        NUM_CANCELLED_count = ('arr_cancelled', 'sum'),
        NUM_DIVERTED_count = ('arr_diverted', 'sum')
    )

    # Step 3: Calculate derived metrics
    group['CANCEL_RATE'] = group['NUM_CANCELLED_count'] / group['TOTAL_FLIGHTS_count']
    group['DIVERT_RATE'] = group['NUM_DIVERTED_count'] / group['TOTAL_FLIGHTS_count']
    group['DELAY_PER_FLIGHT'] = group['TOTAL_ARR_DELAY_count'] / group['TOTAL_FLIGHTS_count']

    # Scale all 4 bad-performance indicators
    scaler = MinMaxScaler()
    metrics = ['AVG_ARR_DELAY_mins', 'CANCEL_RATE', 'DIVERT_RATE', 'DELAY_PER_FLIGHT']
    scaled_vars = ['S_ARR_DELAY_15', 'S_CANCEL', 'S_DIVERT', 'S_DELAY_PER_FLIGHT']
    scaled_metrics = scaler.fit_transform(group[metrics])
    group[scaled_vars] = scaled_metrics

    # Generate composite score with features weighted
    group['COMPOSITE_SCORE'] = (
        .4 * group['S_ARR_DELAY_15'] +
        .3 * group['S_DELAY_PER_FLIGHT'] +
        .2 * group['S_CANCEL'] +
        .1 * group['S_DIVERT']
    )

    top5_worst = group.sort_values('COMPOSITE_SCORE', ascending=False).head(5)
    print("Top 5 worst: ", feature)
    print(top5_worst)

    return group

# Focus on arrival dealys to track airline and airport performance
carrier_group = group_delay_scaled('carrier_name', df).reset_index()
airport_group = group_delay_scaled('airport', df).reset_index()
airport_group.rename(columns={'airport': 'iata_code'}, inplace=True)


# Airport Composite Score Bubble Map

In [ ]:
# Combine lat long data
plt_cols = ['airport_name', 'iata_code', 'latitude_deg', 'longitude_deg', 'COMPOSITE_SCORE']
df_loc = pd.read_csv("/kaggle/input/airport-lat-long/airports_lat_long.csv")
merge_cols = ['iata_code', 'latitude_deg', 'longitude_deg']
df_merged = pd.merge(airport_group, df_loc[merge_cols], on='iata_code', how='left')

# Create bubble map
fig = px.scatter_geo(
        df_merged,
        lat='latitude_deg',
        lon='longitude_deg',
        text='iata_code',
        size='COMPOSITE_SCORE',  # Bubble size based on value
        color='COMPOSITE_SCORE',       # color gradient
        color_continuous_scale='Viridis',  # or 'Plasma', 'Inferno', 'Turbo', etc.
        projection='albers usa',
        title='Bubble Chart of U.S. Airport Average Delay times from 2013- 2023',
        scope='usa'
    )

fig.show()

# ML Model Training and Validation

 Ojbective: Based on the airline, airport, month, and year; how many delays can I expect during that month?


In [ ]:
# Features
feature_1 = ['Month', 'Year', # time 
                'carrier', 'airport', # categorical vals
                'arr_flights' # total flights
                ]

# Not sure if I should be using cancellations or diversions too...
feature_2 = ['Month', 'Year', # time 
                'carrier', 'airport', # categorical vals
                'arr_cancelled', 'arr_diverted', # cancellations and diversions
                'arr_flights' # total flights
                ]

feature_3 = ['Month', 'Year', # time 
                'carrier', 'airport', # categorical vals
                'arr_cancelled', 'arr_diverted'# cancellations and diversions
                ]

feature_4 = ['Month', 'Year', # time 
                'carrier', 'airport' # categorical vals
                ]

feature_set = [feature_1, feature_2, feature_3, feature_4]

for feature in feature_set:

    # Set up features and pre-processor
    cat_cols = ['carrier', 'airport']
    num_cols = [col for col in feature if col not in cat_cols]
    preprocessor = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)],
                                 remainder='passthrough')

    # Set up train and split
    X = df[feature]
    y = df['arr_del15']
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                        random_state=42)
    
    # Select Model
    model = XGBRegressor(
                n_estimators=500,
                max_depth=10,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                objective='reg:squarederror',  # good for count/continuous targets
                random_state=42
            )    


    # Set up pipeline for pre-processing and regression model
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
        ])

    # Fit model
    pipeline.fit(X_train,y_train)

    # Make predictions
    y_pred = pipeline.predict(X_test)

    # Model Validation
    print("using feature set: ", feature)
    print("Y test range: ", y_test.min(), "-", y_test.max())
    print(f"RMSE: {mean_squared_error(y_test, y_pred, squared=False):.2f}")
    print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")
    print(f"R² Score: {r2_score(y_test, y_pred):.3f}")
    
    
    plt.scatter(y_test, y_pred, alpha=0.5)
    plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--')  # ideal line
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.title('Actual vs. Predicted')          
    plt.show()